# 6.4 Your own vectors — what is one row, when a row is "haha"?

6.3's flower cache worked because one row was one flower: a clean photo with a clean label.
Chat data offers no such luck. Almost half of all chat messages are five words or fewer, and
an embedding of "ok" is a vector about "ok".

This notebook follows what happens to most students who embed their chat, one step at a time,
and what to do at each step:

1. **The picture shows nothing.** Embed every message, project, colour by author: a blob, and
   why that is the right answer. (6.4.1)
2. **The tempting rescue.** Ask k-means for three groups, get three groups, and find out what
   they are. (6.4.2)
3. **Choose a better row.** Merge messages into larger units, three ways. (6.4.3)
4. **Check the row with a test you can predict.** Does the second half of a unit belong with
   its first half? (6.4.4)
5. **What "near" means to an embedding.** Topic, not habit. (6.4.5)
6. **A race.** Same task, same classifier, two kinds of vector: which one knows who is
   writing? (6.4.6)
7. **A model somebody else trained, checked.** A sentiment score against the smileys people
   typed themselves. (6.4.7)
8. **Your turn**, with the same functions on your own chat. (6.4.8)

The showcase is the IRC channel `#ubuntu-uk`, where six prolific authors give every check
something to check against.

> Needs `torch`, `sentence-transformers` and `transformers`: `uv sync --extra huggingface`
> once, locally.

In [ ]:
import numpy as np
import pandas as pd
from notebooktester import param
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from transformers import pipeline as hf_pipeline

from goad_toolkit.visualizer import BarPlot, GroupedBarPlot, HeatmapPlot, PlotSettings, ProjectionPlot
from scripts.pipelines import build_irc_pipeline
from scripts.sessionize import fit_session_threshold, merge_messages, sessionize
from wa_analyzer.data import load_own_chat, load_showcase

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts) -> np.ndarray:
    """One 384-number vector per text, from the same small sentence encoder throughout."""
    return np.asarray(EMBEDDER.encode(list(texts), batch_size=64, show_progress_bar=False))

## 6.4.1 A picture that shows nothing

Six of `#ubuntu-uk`'s most active posters, one row per message. Before embedding anything,
read some rows the way the model will get them: one at a time, without context, without a
name.

In [ ]:
irc = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = irc[(irc.channel == "#ubuntu-uk") & (~irc.is_action)].copy()
uk["timestamp"] = (
    pd.to_datetime(uk["date"]) + pd.to_timedelta(uk["hh"], unit="h") + pd.to_timedelta(uk["mm"], unit="m")
)
uk = uk.sort_values(["author", "timestamp"]).reset_index(drop=True)

TOP_AUTHORS = uk.author.value_counts().head(6).index.tolist()
BLOB_SAMPLE = param(1500, test=150)
sample = uk[uk.author.isin(TOP_AUTHORS)].sample(n=BLOB_SAMPLE, random_state=42)
n_words = sample.message.str.count(r"\S+")  # words: runs of non-space characters

print(f"six authors: {', '.join(TOP_AUTHORS)}")
print(f"{len(sample):,} messages, median {n_words.median():.0f} words, "
      f"{(n_words <= 5).mean():.0%} have five words or fewer\n")
for message in sample.message.head(10):
    print(f"   {message}")

Who wrote "!"? Who wrote "like it? based on what?"? You cannot tell, and neither could anyone
else reading them: there is nothing in these lines about their writer. Keep that in mind for
the picture below. An embedding turns every message into 384 numbers, but it has no source
other than the words. More dimensions do not add information the words do not carry.

In [ ]:
X = embed(sample.message)
pca = PCA().fit(X)
coords = pca.transform(X)[:, :2]

settings = PlotSettings(
    figsize=(7, 6),
    title=f"{len(sample):,} messages, six authors, one row per message",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=sample.author.to_numpy(),
                                        palette="tab10", s=10)

print(f"silhouette by author, in all 384 dimensions: {silhouette_score(X, sample.author):.3f}  (0 = no structure)")

A blob, and a silhouette of zero — in all 384 dimensions, not only in the picture. That is the
correct answer, and nothing failed. The encoder puts messages with similar meanings near each
other, and "thanks!" means the same thing whoever types it. If you cannot tell the authors
apart by reading their messages, a model that gets nothing but those messages cannot either.

This happens to most students who embed a chat, and the instinct is to fix the picture. The
better question is the one lesson 1 opened with: **what is one row?** A single message is too
small a unit to carry who wrote it. 6.4.3 changes the row. First, the fix that does not work.

## 6.4.2 The tempting rescue: ask k-means for three groups

A blob is disappointing, and there is a standard move that turns it into a result: run
k-means with K=3 on the picture and colour the points by cluster. Three tidy groups appear.
Before they go in a report, three questions:

1. **How much of the data is in the picture?** 6.1 said to read the scree plot first.
2. **Would the same steps find groups in data that has none?** Shuffle every column on its
   own: each of the 384 columns keeps its values, but which message a value belongs to is
   scrambled, so whatever structure the messages had is gone. Then run the identical steps.
3. **What are the groups?** Check them against something you measured another way.

In [ ]:
cumulative = np.cumsum(pca.explained_variance_ratio_)
print(f"two components carry {cumulative[1]:.1%} of the variance; "
      f"half of it takes {int(np.argmax(cumulative >= 0.5)) + 1} components")

rng = np.random.default_rng(0)
shuffled = np.column_stack([rng.permutation(column) for column in X.T])
shuffled_coords = PCA(n_components=2).fit_transform(shuffled)

groups = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(coords)
shuffled_groups = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(shuffled_coords)
full_groups = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)

print(f"silhouette of three k-means groups, in the picture:      messages {silhouette_score(coords, groups):.2f}, "
      f"shuffled columns {silhouette_score(shuffled_coords, shuffled_groups):.2f}")
print(f"silhouette of three k-means groups, in 384 dimensions:   messages {silhouette_score(X, full_groups):.2f}")

rescue = PlotSettings(
    figsize=(12, 5),
    title="k-means with K=3 returns three groups, whatever it is given",
    subplot_titles=["the messages", "the same numbers, every column shuffled"],
    xlabel="",
    ylabel="",
    max_cols=2,
)
host = ProjectionPlot(rescue)
fig, axes = host.create_figure(n_plots=2)
for ax, points, found in [(axes[0], coords, groups), (axes[1], shuffled_coords, shuffled_groups)]:
    host.plot_on_axes(ProjectionPlot(rescue), ax, coordinates=points,
                      labels=np.array([f"group {g}" for g in found]), palette="Set1", s=10, legend=False)

The first two answers:

1. Two components hold about 7% of the variance, and half of it takes dozens. The picture is
   a shadow of a small part of the data; points that sit together in it can be far apart in
   the rest.
2. The shuffled numbers — no structure, by construction — come apart in three groups too,
   with a silhouette in the picture not far below the real one. And in all 384 dimensions,
   three k-means groups of the real messages score about zero.

So the groups are mostly a property of the picture. The third question says what they are:

In [ ]:
contents = pd.DataFrame({"group": groups, "author": sample.author.to_numpy(), "words": n_words.to_numpy()})
print(contents.groupby("group").words.median().rename("median words per message").to_string())
print()
print(pd.crosstab(contents.group, contents.author, normalize="index").round(2).to_string())

Short, medium and long messages, and every author in every group. k-means did not fail
either: it cut the cloud along the direction that varies most — how much there is in a
message — and handed back three slices, because three is what it was asked for. It returns
K groups on any data, which is why "k-means found three clusters" is never a finding on its
own.

This is lesson 5's trap in a new costume: if you look with a tool that always finds
something, finding something tells you nothing. What does tell you something is what the
three questions did — the share of the data in the picture, the same steps on data without
structure, and a check of the groups against something measured another way. Here that was
a word count; in your project it may be a regex. If the groups show up nowhere but in the
picture, they belong to the picture.

## 6.4.3 Three ways to define one row

Merge consecutive messages from the same author into a unit, three different ways:

1. **Fixed window** — every 15 messages from one author, whatever they happen to be about.
2. **Per-author-per-day** — everything one person said on one calendar day.
3. **Time-gap sessions** — a new unit whenever the gap since that person's last message is
   longer than a threshold. That threshold should not be guessed.

Lesson 4 fit an exponential to the gaps between messages, restricted to gaps under an hour
— the within-burst regime — and read the fit as evidence of *when* people talk in bursts
rather than at a steady rate. The same fit, on this data, gives the number sessionization
needs: the gap length beyond which a next message looks more like a new start than a reply.

In [ ]:
threshold = fit_session_threshold(uk["timestamp"], uk["author"])
print(f"session threshold: {threshold:.0f}s ({threshold / 60:.1f} minutes)")

uk["session_id"] = sessionize(uk, "timestamp", "author", threshold)
subset = uk[uk.author.isin(TOP_AUTHORS)].copy()

K = 15
subset["window_id"] = subset.groupby("author").cumcount() // K
subset["day"] = subset["timestamp"].dt.date

fixed_units = merge_messages(subset, ["author", "window_id"])
daily_units = merge_messages(subset, ["author", "day"])
session_units = merge_messages(subset, ["author", "session_id"])

for name, units in [("fixed-window", fixed_units), ("per-day", daily_units), ("session", session_units)]:
    words = units.message.str.count(r"\S+")
    print(f"{name:>12s}: {len(units):>6,} units, median {words.median():>4.0f} words each")

About a quarter of an hour, read off the fit rather than picked. Three defensible answers to
"what is one row" — and many sessions are still a single message, because people often say
one thing and leave. The next section checks which kind of unit actually holds together.

## 6.4.4 Which row is about one thing?

A good unit is *about something*: one conversation, one topic. What would you expect of such
a unit? That its second half continues its first half. If someone spends ten minutes on a
broken printer, the last five minutes are still about the printer — and you could match the
two halves up even if they were shuffled among hundreds of others.

That gives a check without any labels. Cut every unit in half by time, embed both halves,
and for each first half look for the most similar second half, by cosine (6.3.3). If it finds
its own other half, the unit held together.

Predict the order before running it:

- a **session** is one stretch of talking without a long pause: usually one conversation;
- a **fixed window** of 15 messages is cut wherever message 15 happens to fall, often in the
  middle of a conversation, and can span two;
- a **day** holds every conversation that person had that day, mixed together;
- a **single message**, matched against a later single message from the same session, has
  very little to match on.

Note what "near" means here. For a sentence embedding, two texts are near when they are
about the same thing. That is what this check relies on, and it is also why 6.4.1 found no
authors: being about the same thing is not the same as being written by the same person.

In [ ]:
def self_retrieval_accuracy(frame: pd.DataFrame, group_cols: list[str], min_messages: int = 6,
                            n_eval: int = 400, seed: int = 42,
                            single_message: bool = False) -> tuple[float, int]:
    """Split each unit's messages in half by time, embed both halves, and check how often
    cosine similarity finds the true other half.

    With `single_message=True` each half is one message rather than several joined together
    -- one from early in the unit against one from later -- which is the raw-message baseline.
    """
    sizes = frame.groupby(group_cols).size()
    keys = sizes[sizes >= min_messages].index.tolist()
    rng = np.random.default_rng(seed)
    if len(keys) > n_eval:
        keys = [keys[i] for i in rng.choice(len(keys), size=n_eval, replace=False)]

    first, second = [], []
    grouped = frame.sort_values("timestamp").groupby(group_cols)
    for key in keys:
        grp = grouped.get_group(key)
        half = len(grp) // 2
        if single_message:
            first.append(grp.message.iloc[0])
            second.append(grp.message.iloc[half])
        else:
            first.append(" ".join(grp.message.iloc[:half]))
            second.append(" ".join(grp.message.iloc[half:]))

    A, B = embed(first), embed(second)  # noqa: N806
    A = A / np.linalg.norm(A, axis=1, keepdims=True)  # noqa: N806
    B = B / np.linalg.norm(B, axis=1, keepdims=True)  # noqa: N806
    nearest = (A @ B.T).argmax(axis=1)
    accuracy = float((nearest == np.arange(len(keys))).mean())
    return accuracy, len(keys)

Single raw messages cannot be split in half, so their baseline is the same check with
`single_message=True`: one message from early in a session against one from later in the
same session, rather than a text split down the middle.

In [ ]:
N_EVAL = param(400, test=40)

results = {"single message": self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL,
                                                     single_message=True)}
results["per-day"] = self_retrieval_accuracy(subset, ["author", "day"], n_eval=N_EVAL)
results["fixed-window"] = self_retrieval_accuracy(subset, ["author", "window_id"], n_eval=N_EVAL)
results["session"] = self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL)

print(f"{'unit':>14s}  {'finds its other half':>20s}  {'n units':>8s}   vs. chance")
for name, (acc, n) in results.items():
    print(f"{name:>14s}  {acc:>20.1%}  {n:>8,}   {acc * n:>5.0f}x")

The order you predicted, if you predicted it: sessions find their own other half 29% of the
time, fixed windows 19%, days 13%, single messages 6%. Every one of them is far above chance
— with 400 candidates, a guess finds the right half one time in 400 — but the differences
between them are what the check is for. Days and fixed windows hold more text than a typical
session and still lose, so the amount of text is not what makes a unit match its other half.
Whether the text belongs together is. A fixed window cuts at message 15 whatever is
happening; a day mixes every conversation of that day. The session boundary, read off the
gaps the way §6.4.3 fitted it, is the one that follows the conversation.

## 6.4.5 Near in meaning, not in habit

Two things a merged, embedded corpus is good for: finding what is similar to one thing, and
asking whether "similar" lines up with something you can check — here, who wrote it.

In [ ]:
units = session_units[session_units.n_messages >= 5].reset_index(drop=True)
vectors = embed(units.message)
normed = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

rng = np.random.default_rng(7)
query = rng.integers(0, len(units))
similarity = normed @ normed[query]
ranked = np.argsort(similarity)
nearest, farthest = ranked[-2], ranked[0]  # -1 is the query itself

for label, i in [("QUERY", query), ("NEAREST", nearest), ("FARTHEST", farthest)]:
    print(f"{label} ({units.author.iloc[i]}, cosine={similarity[i]:.2f}):")
    print(f"  {units.message.iloc[i][:200]}")
    print()

The nearest session is about the same kind of thing the query was — both are about squeezing
more out of limited memory, one for swap and one for a GPU — and the farthest is a different
conversation altogether, about sending audio over a network. Cosine similarity on merged
units retrieves by topic, which is what the encoder was trained for and what single messages
could not support.

Look at who wrote them: the query is popey's, the nearest session is ali1234's. Near in
meaning says nothing about the writer, and the next cell checks that for all six.

In [ ]:
print(f"silhouette of the session units by author: {silhouette_score(normed, units.author, metric='cosine'):.3f}")

centroids = pd.DataFrame(
    {author: normed[units.author.to_numpy() == author].mean(axis=0) for author in TOP_AUTHORS}
).T
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)
author_similarity = pd.DataFrame(
    centroids.to_numpy() @ centroids.to_numpy().T, index=TOP_AUTHORS, columns=TOP_AUTHORS
)

settings = PlotSettings(figsize=(6, 5), title="Author centroids, cosine similarity", xlabel="", ylabel="")
fig, ax = HeatmapPlot(settings).plot(data=author_similarity, fmt=".2f")

The sessions do not group by author either: a silhouette of zero, as in 6.4.1, now on units
of dozens of words. And every pair of author averages sits at 0.82–0.94 — high, and barely
different between six different people. That is what a **semantic** embedding is supposed to
do. Everyone in this channel talks about Ubuntu, computers and their day, so everyone's
average points roughly the same way.

5.5 told these same people apart cleanly, with regexes: `:-)` against `:)`, ☻ for one person
and nobody else, dropped apostrophes. That signal is *style* — punctuation and spelling — and
a sentence embedding is trained to ignore it: "dont" and "don't" mean the same thing, so they
should land in the same place. A model measures what it was trained to measure. For "who is
writing", a handful of regexes beat 384 learned dimensions here, because they look at the
right thing.

## 6.4.6 A race: which vector knows who is writing?

6.4.5 suggests that a sentence embedding knows *what* people talk about and little about
*who* they are, and 5.5 found these same people easy to tell apart by *how* they type. Both
are impressions, from a heatmap and from averages. A race turns them into one number each:
give two kinds of vector the same task, the same classifier and the same data, change nothing
but the vector, and see which gets further.

The pieces, each one a decision:

- **The task.** Read one session and name which of the six authors wrote it.
- **The two vectors.** Character-trigram counts from 6.2, which see style — punctuation,
  smileys, spelling — and the sentence embedding from this notebook, which sees meaning.
- **The classifier.** Logistic regression, as in 5.5: one weight per feature per author, an
  author's score is the weighted sum of the features, and the highest score wins. The same
  classifier for both, so a difference comes from the vectors and not from the method.
- **Fair data.** At most 600 sessions per author, so nobody wins by always guessing the
  busiest person. Then 70% to learn from and 30% to score on: sessions the model has never
  seen. Scoring on the training sessions would measure memory, not knowledge.
- **The baseline.** Always guessing the most common author in the test set. With six groups
  of equal size that is one in six.

It is one function, so that 6.4.8 can run the same race on your own chat.

In [ ]:
def race(units: pd.DataFrame, per_author: int = 600,
         seed: int = 42) -> tuple[dict[str, float], LogisticRegression, CountVectorizer]:
    """Name the author of every unit from trigram counts and from an embedding: same classifier,
    same balanced train/test split. Returns the accuracies and the fitted trigram model."""
    shuffled = units.sample(frac=1, random_state=seed)
    balanced = shuffled.groupby("author").head(per_author)
    train, test = train_test_split(balanced, test_size=0.3, stratify=balanced.author, random_state=seed)

    trigrams = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
    by_style = LogisticRegression(max_iter=1000).fit(trigrams.fit_transform(train.message), train.author)
    by_meaning = LogisticRegression(max_iter=1000).fit(embed(train.message), train.author)

    scores = {
        "guess the most common author": float(test.author.value_counts(normalize=True).max()),
        "trigram counts": float(by_style.score(trigrams.transform(test.message), test.author)),
        "sentence embedding": float(by_meaning.score(embed(test.message), test.author)),
    }
    return scores, by_style, trigrams


PER_AUTHOR = param(600, test=80)
scores, by_style, trigrams = race(units, per_author=PER_AUTHOR)

board = pd.DataFrame({"vector": list(scores), "accuracy": list(scores.values())})
settings = PlotSettings(figsize=(8, 3.5),  # ty: ignore[invalid-argument-type]
                        title="Who wrote this session? Accuracy on sessions the model never saw",
                        xlabel="", ylabel="accuracy")
fig, ax = BarPlot(settings).plot(data=board, x="vector", y="accuracy", color="#4c72b0")
for name, accuracy in scores.items():
    print(f"{name:>30s}: {accuracy:.1%}")

Counting characters names the author of about four sessions in five; the embedding gets
about one in two. Both beat guessing (one in six) by a wide margin, and the cheap hand-made
vector beats the learned one by a wide margin too. Why it wins is in its weights, which can
be read: per author, the trigrams that push hardest towards that author.

In [ ]:
def strongest_trigrams(model: LogisticRegression, vectorizer: CountVectorizer, n: int = 8) -> dict[str, list[str]]:
    """The n trigrams that push hardest towards each author. A model with two authors has one
    row of weights, for the second author; the first author's are the same weights negated."""
    vocabulary = vectorizer.get_feature_names_out()
    weights = model.coef_ if len(model.classes_) > 2 else np.vstack([-model.coef_[0], model.coef_[0]])
    return {author: [vocabulary[j] for j in np.argsort(row)[::-1][:n]]
            for author, row in zip(model.classes_, weights)}


for author, strongest in strongest_trigrams(by_style, trigrams).items():
    print(f"{author:12s} {'  '.join(repr(t) for t in strongest)}")

Smileys and spelling. `:-p` and `lol` for diddledan, `:d` and `:>` for daftykins, ☻ and
`nt ` — *dont*, *cant* — for popey, and for MartijnVdS `s: `, `y: `, `o: `: the end of a
nickname followed by a colon: someone being addressed, which was MartijnVdS's role in the
channel in 5.5. These are 5.5's habits, found by the model on its own from raw character counts. The
embedding cannot see them.

That is the result in one sentence: **here, who someone is lives in how they type, and a
sentence embedding measures what they say.** You only find out which of the two your
question needs by racing them on that question.

## 6.4.7 A sentiment score, checked before it is used

One more model somebody else trained: a sentiment classifier that gives every text three
scores, positive, neutral and negative. It is tempting to run it over everything and report
who is the most negative person in the channel. Two questions first.

**What is one row?** A mood belongs to a message — "argh, it crashed again" — not to a
session of seventy words that mixes three moods. So here the row is a single message, the
opposite choice from 6.4.3: the right unit depends on the question.

**Does the score measure mood on this data?** The model learned what "negative" means from
other people's texts, and that need not be what negative means in a support channel. You
check a model like this on cases where you already know the answer, and this channel has
tens of thousands of them: messages where the writer typed their own mood label, a smiley.
Take messages with `:)` or with `:(`, remove the smiley so the model only sees the words,
and compare what it says with what the writer typed. Messages without a smiley come along
as a third group.

In [ ]:
sentiment_model = hf_pipeline(  # ty: ignore[no-matching-overload]
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None, truncation=True, max_length=512,
)

SMILE = r"(?:[:;=]-?[)\]D]|☺|☻)"
FROWN = r"(?:[:;=]-?[(\[]|☹)"


def sentiment_check(messages: pd.Series, n: int, smile: str = SMILE, frown: str = FROWN,
                    seed: int = 0) -> pd.DataFrame:
    """Score up to `n` messages with only a smile, only a frown, and neither — symbols removed,
    at least four words left — and record what the writer typed next to what the model says."""
    has_smile, has_frown = messages.str.contains(smile), messages.str.contains(frown)
    words = messages.str.replace(f"{smile}|{frown}", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    typed = pd.Series(np.select([has_smile & ~has_frown, has_frown & ~has_smile, ~has_smile & ~has_frown],
                                ["typed :)", "typed :(", "no smiley"], "both"), index=messages.index)
    usable = (typed != "both") & (words.str.count(r"\S+") >= 4)
    picked = (pd.DataFrame({"typed": typed[usable], "words": words[usable]})
              .sample(frac=1, random_state=seed).groupby("typed").head(n))
    results = sentiment_model(picked.words.tolist(), batch_size=32)
    scores = pd.DataFrame([{item["label"]: item["score"] for item in result} for result in results],
                          index=picked.index)
    picked["model says"] = scores[["positive", "neutral", "negative"]].idxmax(axis=1)
    return picked


SENTIMENT_SAMPLE = param(600, test=40)
checked = sentiment_check(uk.message, n=SENTIMENT_SAMPLE)
agreement = pd.crosstab(checked.typed, checked["model says"], normalize="index")
print(agreement.round(2).to_string())

shares = agreement.reset_index().melt(id_vars="typed", var_name="model says", value_name="share")
settings = PlotSettings(figsize=(9, 4), title="What the model says, by the smiley the writer typed",
                        xlabel="", ylabel="share of messages")
fig, ax = GroupedBarPlot(settings).plot(data=shares, x="typed", y="share", hue="model says",
                                        order=["typed :(", "no smiley", "typed :)"],
                                        hue_order=["negative", "neutral", "positive"],
                                        palette={"negative": "crimson", "neutral": "#bbbbbb",
                                                 "positive": "steelblue"})

A model that read mood perfectly would call every `:(` message negative and every `:)`
message positive. One that read nothing would give all three groups the same shares. This
one is much closer to the second. Of the messages typed with `:(` it calls 57% negative, of
those typed with `:)` 44%, and of plain messages 47%. It picks up a little of the mood, and
it calls almost half of all chat negative; read some and you find people describing software
that does not work, which is what a support channel is for, not a mood.

What that means for the per-person table you were about to make: a difference of a few
points in average negativity between two people is smaller than the model's own confusion,
and would mostly measure who talks about broken things. The honest sentence on this channel
is *"the sentiment model agrees only weakly with the writers' own smileys, so it is not used
to compare people"*. That is a result, and it took one table to get it.

In a chat between friends it may do better: more of the text is about feelings, and the
model is multilingual, so Dutch works. Your chat has its own labels to check it against —
emoji — and 6.4.8 runs the same check. If it fails there too, write the same kind of sentence.

## 6.4.8 Your turn

The same steps on your own chat, one cell each. Run a cell, look at what it prints, and
write one sentence per question in your own words. Those sentences, with their numbers, are
the result; the plots are how you got there.

1. **The blob.** *Do single messages from your chat separate by author? Could you tell your
   friends apart from one message yourself?*
2. **Your row.** *What is your session threshold in minutes, and does it match how your chat
   feels? How many sessions have five or more messages, and for whom?*
3. **The half-session check.** *Do your sessions hold together better than chance?*
4. **The race**, if at least two people have twenty such sessions. *Which vector names the
   author better, and what do the trigram weights say each person does?*
5. **The sentiment check**, with emoji instead of smileys, if your chat has enough of both
   kinds. *Does the model agree with the emoji people typed?*

A null result is a result: "single messages from my chat do not separate by author
(silhouette 0.01)" is a finished sentence.

In [ ]:
own = load_own_chat().sort_values(["author", "timestamp"]).reset_index(drop=True)
own_sample = own.sample(n=min(len(own), BLOB_SAMPLE), random_state=42)
own_X = embed(own_sample.message)

mine = PlotSettings(figsize=(7, 6), title="Your chat, one row per message", xlabel="", ylabel="")
fig, ax = ProjectionPlot(mine).plot(coordinates=PCA(n_components=2).fit_transform(own_X),
                                    labels=own_sample.author.to_numpy(), palette="tab10", s=10)
print(f"silhouette by author, in 384 dimensions: {silhouette_score(own_X, own_sample.author):.3f}")

In [ ]:
own_threshold = fit_session_threshold(own["timestamp"], own["author"])
own["session_id"] = sessionize(own, "timestamp", "author", own_threshold)
own_units = merge_messages(own, ["author", "session_id"])
own_checkable = own_units[own_units.n_messages >= 5].reset_index(drop=True)

print(f"your session threshold: {own_threshold / 60:.1f} minutes")
print(f"{len(own_units):,} sessions, {len(own_checkable):,} with five or more messages:")
print(own_checkable.author.value_counts().to_string())

In [ ]:
own_acc, own_n = self_retrieval_accuracy(own, ["author", "session_id"], min_messages=5, n_eval=N_EVAL)
if own_n:
    print(f"{own_acc:.1%} of first halves find their own second half (n={own_n}, chance {1 / own_n:.1%})")
else:
    print("No sessions with five or more messages: the check has nothing to work on.")

In [ ]:
enough = own_checkable.author.value_counts()
racers = own_checkable[own_checkable.author.isin(enough[enough >= 20].index)]
if racers.author.nunique() >= 2:
    own_scores, own_style, own_trigrams = race(racers, per_author=PER_AUTHOR)
    for name, accuracy in own_scores.items():
        print(f"{name:>30s}: {accuracy:.1%}")
    print()
    for author, strongest in strongest_trigrams(own_style, own_trigrams).items():
        print(f"{author:12s} {'  '.join(repr(t) for t in strongest)}")
else:
    print("Fewer than two people with twenty sessions of five or more messages: no race on this chat.")

In [ ]:
EMOJI_SMILE = "[😀😃😄😁😊🙂😍🥰😘]"
EMOJI_FROWN = "[😢😭😞😔🙁😟😩😠]"
own_checked = sentiment_check(own.message, n=SENTIMENT_SAMPLE, smile=EMOJI_SMILE, frown=EMOJI_FROWN)
if {"typed :)", "typed :("} <= set(own_checked.typed):
    print(pd.crosstab(own_checked.typed, own_checked["model says"], normalize="index").round(2).to_string())
    print("\nmessages per group:", own_checked.typed.value_counts().to_dict())
else:
    print("Your chat lacks one of the two kinds of emoji (at least four words besides it): no check possible.")

## What to carry

1. **A picture can only show what the rows contain.** If you cannot tell who wrote a message
   by reading it, an embedding of it cannot either; the blob was the right answer.
2. **A method that always returns an answer is not evidence.** k-means returns K groups on
   anything. Ask how much of the data the picture holds, run the same steps on data without
   structure, and check the groups against something you measured another way.
3. **What one row is depends on the question.** Sessions for topics, single messages for
   moods. Check the choice with a test whose outcome you can predict.
4. **An embedding measures what it was trained on**: meaning, not habit. For who is writing,
   counting characters beats it, and a race on the same task is how you find out.
5. **A model somebody else trained uses somebody else's definitions.** Check it on cases
   where you know the answer before you use it to compare people.